# Beatrix — early-life curriculum
Her school notebook. Plan of record: `claude-mind/history/plans/2026-08-15_early_life_curriculum.md`.

Nine stages, ~8.8B tokens, resume-first: open this notebook any day, run **Cell 1** then **Cell 3** with whatever hours you have, and walk away. Stage boundaries fire their own instruments. Cell 2 runs ONCE (baseline, both cores). Cell 4 renders growth anytime.

Laws wired in: no chat template / no identity in any training row · causal-test holdout families never reach a training row (bAbI 16+19, depth-5 chains, 3-digit subtraction) · every stage vals on the same fineweb-2013 holdout (gauge continuity) · probes carry the stage-specific measurement.

In [ ]:
%pip install -q "geolip-alephllm @ git+https://github.com/AbstractEyes/alephllm@0e3b8ddced249a046d75d18e3192a518ef865cd8" "amoe-lora @ git+https://github.com/AbstractEyes/amoe-lora@7e1baba9c0bb1cc38815e4424d969b80ae5a315d"
import os, torch
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
assert torch.cuda.is_available(), "GPU runtime required"
import geolip.alephllm as al
print("geolip", al.__version__, "·", torch.cuda.get_device_name(0))

## Cell 2 — instrument gate (run ONCE, before any training)
Probes P0–P8 on **both** cores — the locked 51,882 and the annealed 58,664 — so every growth curve has a time-zero. Ships the baseline report to the training repo. The head surgery (fold the fossilized gamma gate into W_s; verified semantic no-op, max|diff| 2.4e-07) is applied to the eval copies here and to the live run in Cell 3.

In [ ]:
import json, torch
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download, HfApi
from geolip.alephllm.presets import get_preset
from geolip.alephllm.model.alephlm import AlephLM
from geolip.alephllm.data import build_tokenizer
from geolip.alephllm.data import curriculum as C
from geolip.alephllm.train import probes

REPO = "AbstractPhil/alephllm-mini-beatrix-training"
BASE_STEP = 58664          # the curriculum base (51882 = the clean lock)
preset = get_preset("mini-beatrix-1")
tok = build_tokenizer(preset.model.tokenizer)
baseline = {}
for step in (51882, BASE_STEP):
    ck = hf_hub_download(REPO, f"mini-beatrix-1/checkpoints/step_{step:08d}.safetensors")
    m = AlephLM(preset.model); m.load_state_dict(load_file(ck)); m.cuda().eval()
    fold = C.fold_head_gate(m)          # no-op when gamma already 1.0
    res = probes.run_all(m, tok, "cuda")
    baseline[str(step)] = {"probes": res, "head_fold": fold}
    print(f"=== step {step:,} (gamma was {fold['gamma_before']:.6f}) ===")
    print(probes.report(res)); print()
    del m; torch.cuda.empty_cache()
with open("baseline.json", "w", encoding="utf-8") as f:
    json.dump(baseline, f, indent=1, default=float)
HfApi(token=HF_TOKEN).upload_file(path_or_fileobj="baseline.json",
    repo_id=REPO, path_in_repo="mini-beatrix-1/reports/curriculum/baseline.json",
    commit_message="curriculum baseline: probes P0-P8 on both cores at t=0")
print("baseline shipped -> reports/curriculum/baseline.json")

## Cell 3 — the school day (re-run every session)
Resumes wherever she is. Each `train()` call is capped at the current stage's remaining tokens, so sessions stop **exactly at boundaries**; the boundary fires probes + toggle ledger + head-election gauge and ships the report. Set `MAX_HOURS` to your session budget.

In [ ]:
import json, time, torch
from huggingface_hub import HfApi
from geolip.alephllm import prepare
from geolip.alephllm.data import curriculum as C
from geolip.alephllm.data.streams import build_stream
from geolip.alephllm.train import probes, instruments

MAX_HOURS = 8.0
REPO = "AbstractPhil/alephllm-mini-beatrix-training"

run = prepare("mini-beatrix-1", hf_token=HF_TOKEN)
added = C.append_curriculum_phases(run.manifest)
if added:
    print(f"curriculum registered: {added} stages appended to the manifest")
if float(run.raw_model.head.gamma.item()) != 1.0:
    info = C.fold_head_gate(run.raw_model)
    print(f"head gate folded: gamma {info['gamma_before']:.6f} -> 1.0 "
          f"(||W_s||={info['w_s_norm']:.4f}; Muon re-elects from here)")

def boundary_instruments(run, tag):
    m, tok = run.raw_model, run.tokenizer
    res = probes.run_all(m, tok, run.device)
    vs = build_stream("fineweb-edu", tok, run.cfg.context,
                      run.tc.micro_batch, seed=run.tc.seed + 9999, role="val")
    val = [vs.next_batch().to(run.device) for _ in range(2)]
    led = instruments.toggle_ledger(m, val)
    ws = float(m.head.w_s.weight.norm().item())
    print(f"== boundary {tag} · step {run.step:,} ==")
    print(probes.report(res))
    print(f"  fineweb gauge: full {led['bpb_full']:.4f} · "
          f"head_off {led['toggle_head_aleph_off']:+.4f} · "
          f"hub_off {led.get('toggle_hub_off', 0):+.4f} · ||W_s||={ws:.3f}")
    rep = {"tag": tag, "step": run.step, "probes": res, "ledger": led,
           "w_s_norm": ws}
    name = f"boundary_{tag}_step{run.step}.json"
    with open(name, "w", encoding="utf-8") as f:
        json.dump(rep, f, indent=1, default=float)
    HfApi(token=HF_TOKEN).upload_file(path_or_fileobj=name, repo_id=REPO,
        path_in_repo=f"mini-beatrix-1/reports/curriculum/{name}",
        commit_message=f"curriculum boundary report: {tag}")

t_end = time.time() + MAX_HOURS * 3600
while time.time() < t_end:
    ph = run.manifest.current_phase()
    if ph is None:
        print("curriculum COMPLETE — all stages done."); break
    remaining = int(ph["planned_tokens"]) - int(ph.get("tokens_done", 0))
    hours_left = (t_end - time.time()) / 3600
    print(f"[school] stage {ph['name']} · {remaining/1e9:.3f}B to go · "
          f"session budget {hours_left:.2f}h")
    run.train(max_tokens=remaining, max_hours=hours_left)
    ph_after = run.manifest.current_phase()
    if ph_after is None or ph_after["name"] != ph["name"]:
        boundary_instruments(run, tag=ph["name"])
    else:
        print("[school] session budget reached mid-stage — resume next "
              "session, she keeps her place.")
        break

## Cell 4 — growth report (run anytime)
The childhood so far: every boundary's probe accuracies against the baseline, holdout families marked, head-election trace.

In [ ]:
import json
from huggingface_hub import HfApi, hf_hub_download
REPO = "AbstractPhil/alephllm-mini-beatrix-training"
api = HfApi()
files = [f for f in api.list_repo_files(REPO)
         if f.startswith("mini-beatrix-1/reports/curriculum/")]
reports = []
for f in sorted(files):
    d = json.load(open(hf_hub_download(REPO, f), encoding="utf-8"))
    if "baseline" in f:
        for step, r in d.items():
            reports.append((f"t0@{step}", int(step), r["probes"], None))
    else:
        reports.append((d["tag"], d["step"], d["probes"], d.get("w_s_norm")))
if not reports:
    print("no reports yet — run Cell 2 (baseline) first")
else:
    suites = sorted(reports[0][2])
    w = max(len(t) for t, *_ in reports) + 2
    print("".ljust(w) + "".join(s.replace("_", " ")[:14].ljust(15)
                                for s in suites))
    for tag, step, pr, ws in reports:
        row = "".join(f"{pr[s]['acc']:.3f}".ljust(15) for s in suites)
        tail = f"  ||W_s||={ws:.3f}" if ws is not None else ""
        print(tag.ljust(w) + row + tail)

## Cell 5 — arm collectives (boundary causal test)
Arrives with geolip 0.6.1 + the amoe harness pass. The holdout families (bAbI 16/19, depth-5 chains, 3-digit subtraction) are **already excluded from every training row**, and the probe batteries already gauge them — so any boundary she passes before this cell lands can be tested retroactively against stored checkpoints (retro-sweep law). Nothing is lost by training first.